# Phase 3 Architecture A v3 — Single-Stage Ordinal & Cosine KNN Features

Upgrades from v2:
1. **Ordinal-Aware Modeling for D1-D5** — trains 4 binary threshold classifiers per dimension using Frank & Hall binary decomposition
2. **Continuous Expected Score Calculation** — derived tier is computed from the probability-weighted expected D-scores, preventing boundary rounding issues
3. **Nearest-Neighbor Cosine Features** — uses training set semantic neighbors to supply context anchors (`nn_tier_mean`, `nn_d1_mean`, etc.) inside each fold

Kept from v2:
- BGE-base-en-v1.5 embeddings (768d) with custom prefix
- PCA = 60
- 50 handcrafted features
- Balanced sample weights

In [1]:
# Colab setup
!pip install -q xgboost sentence-transformers scikit-learn

In [2]:
import ast
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5
PCA_COMPONENTS = 60

## Load Dataset

In [3]:
DATA_PATH = '/content/prompt_classifier_phase3_v1_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(2421, 29)


,id,prompt,intent,task_type,reasoning_chain_detected,d1,d2,d3,d4,d5,...,prompting_techniques,prompt_type,phrasing_style,domain,source,augmentation_group,intent_d1_mismatch_flag,boundary_t12_flag,boundary_t23_flag,original_row_flag
0,NaN,Imagine you are a science fiction author renow...,SYNTHETIC,generation,True,0.75,0.50,0.50,0.0,0.0,...,"['ROLE_PROMPTING', 'TREE_OF_THOUGHTS']",CREATIVE_WRITING,NaN,NaN,phase2,NaN,False,False,False,True
1,NaN,You are an expert photography tutor. I want to...,ANALYTICAL,generation,True,0.50,0.75,0.50,0.0,0.0,...,['CODE_PROMPTING'],CODE_EXPLANATION,NaN,NaN,phase2,NaN,False,True,False,True
2,NaN,You are a leading neuroscientist specializing ...,SYNTHETIC,reasoning,True,0.75,0.75,0.75,0.5,0.5,...,"['ROLE_PROMPTING', 'CHAIN_OF_THOUGHT']",CONVERSATIONAL,NaN,NaN,phase2,NaN,True,False,True,True
3,NaN,I want to understand the basic human emotions....,FACTUAL,generation,False,0.00,0.50,0.50,0.0,0.0,...,"['CHAIN_OF_THOUGHT', 'CONTEXTUAL_PROMPTING']",COMPARISON,NaN,NaN,phase2,NaN,False,False,False,True
4,NaN,Here are examples of competitive exclusion. Ex...,SYNTHETIC,reasoning,True,0.75,0.50,0.50,0.0,0.5,...,['ONE_SHOT_FEW_SHOT'],PROGRAMMING_CODE_GENERATION,NaN,NaN,phase2,NaN,True,False,False,True


## Dataset Validation & Cleanup

In [4]:
SCORE_COLS = ['d1', 'd2', 'd3', 'd4', 'd5']
VALID_SCORES = [0.0, 0.25, 0.5, 0.75, 1.0]
SCORE_TO_CLASS = {score: idx for idx, score in enumerate(VALID_SCORES)}
CLASS_TO_SCORE = {idx: score for score, idx in SCORE_TO_CLASS.items()}

DIMENSION_LABELS = {
    'd1': 'Semantic Complexity',
    'd2': 'Domain Specificity',
    'd3': 'Output Formality',
    'd4': 'Research Dependency',
    'd5': 'Context Requirement',
}

D1_TO_INTENT = {
    0.00: 'FACTUAL',
    0.25: 'FACTUAL',
    0.50: 'ANALYTICAL',
    0.75: 'SYNTHETIC',
    1.00: 'STRATEGIC',
}


def normalize_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() == 'true'


def complexity_score_from_dims_frame(frame):
    return (
        frame['d1'] * 0.35 +
        frame['d2'] * 0.20 +
        frame['d3'] * 0.20 +
        frame['d4'] * 0.15 +
        frame['d5'] * 0.10
    )


def tier_from_score(score):
    if score < 0.40:
        return 'T1'
    if score < 0.70:
        return 'T2'
    return 'T3'


# --- Defensive cleanup ---
df['expected_intent'] = df['d1'].map(D1_TO_INTENT)
misaligned_count = int((df['intent'] != df['expected_intent']).sum())
if misaligned_count > 0:
    print(f'Fixing {misaligned_count} D1<->intent misalignments')
    df['intent'] = df['expected_intent']
else:
    print('D1<->intent alignment: OK')
df.drop(columns=['expected_intent'], inplace=True)

formatting_count = int((df['task_type'] == 'formatting').sum())
df.loc[df['task_type'] == 'formatting', 'task_type'] = 'generation'
sparql_count = int((df['task_type'] == 'sparql_generation').sum())
df.loc[df['task_type'] == 'sparql_generation', 'task_type'] = 'generation'

df['reasoning_chain_detected'] = df['reasoning_chain_detected'].apply(normalize_bool)
df['computed_complexity_score'] = complexity_score_from_dims_frame(df)
df['computed_tier'] = df['computed_complexity_score'].apply(tier_from_score)

dup_count = int(df['prompt'].duplicated().sum())
if dup_count:
    df = df.drop_duplicates(subset='prompt', keep='first').reset_index(drop=True)

print(f'Formatting merged: {formatting_count}, sparql merged: {sparql_count}')
print(f'Tier mismatches: {(df["computed_tier"] != df["tier"]).sum()}')
print(f'Duplicates removed: {dup_count}')
print(f'Final rows: {len(df)}')
print(f'\nTier: {df["tier"].value_counts().sort_index().to_dict()}')
print(f'Intent: {df["intent"].value_counts().to_dict()}')
print(f'Task type: {df["task_type"].value_counts().to_dict()}')

D1<->intent alignment: OK
Formatting merged: 10, sparql merged: 8
Tier mismatches: 0
Duplicates removed: 0
Final rows: 2421

Tier: {'T1': 970, 'T2': 1072, 'T3': 379}
Intent: {'SYNTHETIC': 856, 'ANALYTICAL': 791, 'FACTUAL': 577, 'STRATEGIC': 197}
Task type: {'reasoning': 1314, 'generation': 800, 'summarisation': 120, 'classification': 100, 'coding': 87}


## 50 Hand-Crafted Features

In [5]:
ARTIFACT_TERMS = ['csv', 'json', 'pdf', 'log', 'yaml', 'yml', 'xlsx', 'docx', 'transcript', 'diagram']
CLOUD_PROVIDERS = ['aws', 'azure', 'gcp', 'google cloud', 'oci']
SYSTEMS = ['salesforce', 'servicenow', 'jira', 'workday', 'sap', 'snowflake', 'databricks', 'okta', 'hubspot', 'github', 'gitlab']
FRAMEWORKS = ['itil', 'finops', 'togaf', 'owasp', 'dora', 'nist', 'hipaa', 'soc 2', 'soc2', 'gdpr', 'iso 27001', 'pci-dss', 'pci dss']
VENDOR_TOOLS = sorted(set(CLOUD_PROVIDERS + SYSTEMS + ['openai', 'anthropic', 'bedrock', 'terraform', 'kubernetes', 'docker', 'jenkins', 'splunk']))

DOMAIN_BUCKETS = {
    'cloud': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'budget', 'chargeback', 'showback'],
    'security': ['security', 'vulnerability', 'iam', 'zero trust', 'soc'],
    'devops': ['devops', 'ci/cd', 'pipeline', 'sre', 'deployment'],
    'data': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai': ['ai', 'llm', 'genai', 'machine learning', 'model'],
    'hr': ['hr', 'employee', 'talent', 'workforce', 'recruiting'],
    'supply': ['supply chain', 'inventory', 'procurement', 'logistics'],
}

D1_COMPLEXITY_TERMS = [
    'strategic', 'cross-domain', 'enterprise-wide', 'synthesize', 'multi-cloud',
    'governance', 'architecture', 'framework', 'transformation', 'lifecycle',
    'holistic', 'end-to-end', 'migration', 'orchestration',
]

SIMPLE_FACTUAL_PATTERNS = [
    r'^what is\b', r'^define\b', r'^who is\b', r'^when\b',
    r'^list\b', r'^name\b', r'\bwhat does .{1,30} mean\b',
]

D2_DOMAIN_TERMS = [
    'kubernetes', 'terraform', 'sagemaker', 'databricks', 'snowflake',
    'cicd', 'ci/cd', 'finops', 'mlops', 'devsecops', 'apigee',
    'oauth', 'saml', 'oidc', 'vpc', 'subnet', 'iam',
]


def count_terms_safe(text, terms):
    return sum(1 for term in terms if term in text)


def any_terms_safe(text, terms):
    return any(term in text for term in terms)


def get_style_at(phrasing_styles, i):
    if phrasing_styles is None:
        return None
    try:
        value = phrasing_styles.iloc[i]
    except AttributeError:
        value = phrasing_styles[i]
    return None if pd.isna(value) else str(value).strip().lower()


def handcrafted_features(prompts, phrasing_styles=None):
    rows = []
    for i, prompt in enumerate(prompts):
        text = str(prompt)
        lower = text.lower()
        words = re.findall(r'\b\w+\b', lower)
        unique_words = set(words)
        sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
        lines = [line for line in text.splitlines() if line.strip()]
        style = get_style_at(phrasing_styles, i)

        row = {}

        # Text statistics: 6
        row['char_len'] = len(text)
        row['word_count'] = len(words)
        row['sentence_count'] = max(1, len(sentences))
        row['avg_word_len'] = float(np.mean([len(w) for w in words])) if words else 0.0
        row['unique_word_ratio'] = len(unique_words) / max(1, len(words))
        row['line_count'] = len(lines)

        # D5 Context Requirement: 4
        row['has_attachment'] = int(any_terms_safe(lower, ['uploaded', 'attached', 'provided file', 'document below', 'context below', 'see below']))
        row['provided_artifact_count'] = count_terms_safe(lower, ARTIFACT_TERMS)
        row['large_context_signal'] = int(any_terms_safe(lower, ['across all', 'entire', 'all of our', 'company-wide', 'large context', 'full document']))
        row['multi_document_signal'] = int(any_terms_safe(lower, ['multiple', 'all the', 'each of the', 'various', 'several documents', 'set of files']))

        # D3 Output Formality: 4
        row['has_formal_deliverable'] = int(any_terms_safe(lower, ['report', 'brief', 'proposal', 'specification', 'whitepaper', 'requirements doc']))
        row['has_report_package'] = int(any_terms_safe(lower, ['appendix', 'table of contents', 'risk register', 'executive summary', 'roadmap', 'implementation plan']))
        row['has_long_output_signal'] = int(any_terms_safe(lower, ['comprehensive', 'detailed', 'thorough', 'in-depth', 'end-to-end']))
        row['structured_section_count'] = count_terms_safe(lower, ['executive summary', 'timeline', 'roadmap', 'risk register', 'assumptions', 'recommendations', 'next steps', 'success metrics'])

        # D1 Semantic Complexity: 3
        row['has_scope_words'] = int(any_terms_safe(lower, ['strategic', 'cross-domain', 'enterprise-wide', 'synthesize', 'multi-cloud', 'governance']))
        row['action_verb_count'] = count_terms_safe(lower, ['build', 'design', 'evaluate', 'integrate', 'optimize', 'develop', 'assess', 'recommend', 'compare'])
        row['multi_stage_signal'] = int(bool(re.search(r'\bphase\b|\bstage\b|\bstep\s*1\b|\bmilestone\b|\bsequentially\b|\bfirst\b.*\bthen\b', lower)))

        # D2 Domain Specificity: 4
        row['has_compliance'] = int(any_terms_safe(lower, ['nist', 'hipaa', 'soc2', 'soc 2', 'gdpr', 'iso 27001', 'pci-dss', 'pci dss', 'compliance']))
        row['cloud_providers_mentioned'] = count_terms_safe(lower, CLOUD_PROVIDERS)
        row['systems_mentioned'] = count_terms_safe(lower, SYSTEMS)
        row['domain_framework_count'] = count_terms_safe(lower, FRAMEWORKS)

        # D4 Research Dependency: 5
        row['external_data_score'] = count_terms_safe(lower, ['market research', 'industry report', 'analyst', 'third-party', 'external data', 'latest', 'current'])
        row['has_time_reference'] = int(bool(re.search(r'\b20\d{2}\b|\bfy\d{2}\b|\bthis quarter\b|\blatest\b|\bcurrent\b|\brecent\b|\btoday\b|\bnow\b', lower)))
        row['vendor_tool_count'] = count_terms_safe(lower, VENDOR_TOOLS)
        row['has_market_terms'] = int(any_terms_safe(lower, ['competitor', 'market share', 'tam', 'sam', 'som', 'benchmark', 'industry trend']))
        row['has_cost_comparison'] = int(any_terms_safe(lower, ['pricing', 'tco', 'roi', 'showback', 'chargeback', 'cheapest']) or 'cost analysis' in lower)

        # Boundary/Risk: 3
        row['has_comparison'] = int(any_terms_safe(lower, ['compare', 'versus', 'tradeoff']) or any(phrase in lower for phrase in [' vs ', 'difference between']))
        row['stakeholder_mentions'] = count_terms_safe(lower, ['ceo', 'cto', 'cio', 'cfo', 'board', 'leadership', 'management', 'executive'])
        row['risk_language'] = count_terms_safe(lower, ['risk', 'threat', 'vulnerability', 'mitigation', 'breach', 'exposure', 'audit'])

        # Phase 2 Intent / Reasoning Chain: 5
        row['has_role_prompt'] = int(bool(re.search(r'\byou are\b|\bact as\b|\bassume the role\b', lower)))
        row['has_step_request'] = int(bool(re.search(r'\bstep[- ]by[- ]step\b|\bfirst\b.*\bthen\b|\bsequentially\b', lower)))
        row['has_chain_of_thought'] = int(bool(re.search(r'\bthink through\b|\breason about\b|\blet.s think\b|\bchain of thought\b|\bwalk me through\b', lower)))
        if '?' not in text:
            row['question_complexity'] = 0
        elif any(phrase in lower for phrase in ['what should', 'design a']) or any_terms_safe(lower, ['recommend', 'propose', 'strategy']):
            row['question_complexity'] = 3
        elif any_terms_safe(lower, ['why', 'how', 'compare', 'analyze', 'evaluate', 'assess']):
            row['question_complexity'] = 2
        else:
            row['question_complexity'] = 1
        row['multi_domain_count'] = sum(1 for bucket_terms in DOMAIN_BUCKETS.values() if any_terms_safe(lower, bucket_terms))

        # Phase 2 Task Type: 5
        row['has_code_block'] = int('```' in text)
        row['has_output_format'] = int(bool(re.search(r'\bin json\b|\bas a table\b|\bformat as\b|\bcsv output\b|\bin yaml\b|\bas markdown\b|\bstrict yaml\b|\bstrict json\b', lower)))
        row['has_creative_language'] = int(any_terms_safe(lower, ['imagine', 'creative', 'story', 'compose', 'fictional']) or 'write a' in lower)
        row['has_classification_request'] = int(any_terms_safe(lower, ['classify', 'categorize', 'label']) or any(phrase in lower for phrase in ['which category', 'sort into']))
        row['enumeration_signal'] = int(bool(re.search(r'\blist\b|\btop \d+\b|\benumerate\b|\bbullet point\b|\brank\b', lower)))

        # v5 D1/D2-specific signals: 8
        row['d1_strategic_signal_count'] = count_terms_safe(lower, D1_COMPLEXITY_TERMS)
        row['d1_simple_factual_signal'] = int(any(re.search(pattern, lower) for pattern in SIMPLE_FACTUAL_PATTERNS))
        row['d1_multi_constraint_count'] = count_terms_safe(lower, ['include', 'cover', 'consider', 'account for', 'must'])
        row['d1_solution_design_signal'] = int(any_terms_safe(lower, ['design', 'architect', 'plan', 'strategy', 'roadmap']))
        row['d2_domain_term_count'] = count_terms_safe(lower, D2_DOMAIN_TERMS)
        row['d2_acronym_count'] = len(re.findall(r'\b[A-Z]{2,6}\b', text))
        row['d2_vendor_or_framework_signal'] = int(row['vendor_tool_count'] > 0 or row['domain_framework_count'] > 0)
        row['d2_generic_prompt_signal'] = int(row['d2_domain_term_count'] == 0 and row['cloud_providers_mentioned'] == 0 and row['systems_mentioned'] == 0)

        # Phrasing style: 3
        row['phrasing_explicit'] = int(style == 'explicit')
        row['phrasing_implicit'] = int(style == 'implicit')
        row['phrasing_vague'] = int(style == 'vague')

        rows.append(row)

    feature_df = pd.DataFrame(rows).fillna(0)
    expected_features = 50
    if feature_df.shape[1] != expected_features:
        raise ValueError(f'Expected {expected_features} hand-crafted features, got {feature_df.shape[1]}')
    return feature_df

hand_df = handcrafted_features(df['prompt'], df.get('phrasing_style'))
print('Hand-crafted feature shape:', hand_df.shape)
hand_df.head()

Hand-crafted feature shape: (2421, 50)


,char_len,word_count,sentence_count,avg_word_len,unique_word_ratio,line_count,has_attachment,provided_artifact_count,large_context_signal,multi_document_signal,...,d1_simple_factual_signal,d1_multi_constraint_count,d1_solution_design_signal,d2_domain_term_count,d2_acronym_count,d2_vendor_or_framework_signal,d2_generic_prompt_signal,phrasing_explicit,phrasing_implicit,phrasing_vague
0,1058,158,11,5.506329,0.696203,4,0,1,0,0,...,0,2,0,0,0,1,0,0,0,0
1,531,82,9,5.146341,0.707317,1,0,1,0,0,...,0,1,0,0,1,0,1,0,0,0
2,525,70,5,6.371429,0.814286,1,0,1,0,0,...,0,2,0,0,0,0,1,0,0,0
3,237,38,4,5.078947,0.921053,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,713,107,12,5.364486,0.644860,1,0,1,0,0,...,0,1,1,0,0,0,1,0,0,0


## Embeddings — BGE-base-en-v1.5

In [6]:
EMBEDDING_MODEL_NAME = 'BAAI/bge-base-en-v1.5'
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

prompts = df['prompt'].astype(str).tolist()
prompts_for_encoding = ['Represent this sentence: ' + p for p in prompts]

embeddings = embedding_model.encode(
    prompts_for_encoding,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print('Embedding model:', EMBEDDING_MODEL_NAME)
print('Embedding shape:', embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Embedding model: BAAI/bge-base-en-v1.5
Embedding shape: (2421, 768)


## Nearest-Neighbors Cosine Feature Helper

Extract local context stats using nearest cosine neighbors inside the training subset.

In [7]:
def construct_nn_features(embeddings_part, knn_model, train_df, is_train=False):
    distances, indices = knn_model.kneighbors(embeddings_part)
    if is_train:
        # skip self index
        indices = indices[:, 1:]
        distances = distances[:, 1:]
    else:
        # take first 5
        indices = indices[:, :-1]
        distances = distances[:, :-1]

    train_tier_numeric = np.array([{'T1': 0, 'T2': 1, 'T3': 2}[t] for t in train_df['tier'].values])
    train_d1 = train_df['d1'].values
    train_d2 = train_df['d2'].values

    feats = []
    for i in range(len(indices)):
        idxs = indices[i]
        dists = distances[i]
        feats.append([
            train_tier_numeric[idxs].mean(),
            train_tier_numeric[idxs].std(),
            train_d1[idxs].mean(),
            train_d1[idxs].std(),
            train_d2[idxs].mean(),
            train_d2[idxs].std(),
            dists.mean(),
        ])
    return np.array(feats)

## Feature Matrix Helpers with KNN Pipeline

In [8]:
def fit_shared_transformers(train_embeddings, train_hand_features, train_df):
    pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    train_emb_pca = pca.fit_transform(train_embeddings)

    knn_model = NearestNeighbors(n_neighbors=6, metric='cosine')
    knn_model.fit(train_embeddings)
    train_knn_feats = construct_nn_features(train_embeddings, knn_model, train_df, is_train=True)

    train_raw = np.hstack([train_emb_pca, train_hand_features, train_knn_feats])
    scaler = StandardScaler()
    train_X = scaler.fit_transform(train_raw)
    return pca, scaler, knn_model, train_X


def transform_shared_features(embeddings_part, hand_features_part, pca, scaler, knn_model, train_df):
    emb_pca = pca.transform(embeddings_part)
    knn_feats = construct_nn_features(embeddings_part, knn_model, train_df, is_train=False)
    raw = np.hstack([emb_pca, hand_features_part, knn_feats])
    return scaler.transform(raw)


def fit_full_feature_matrix():
    pca, scaler, knn_model, X_full = fit_shared_transformers(embeddings, hand_df.values, df)
    print('PCA variance explained:', round(float(pca.explained_variance_ratio_.sum()), 4))
    print('Feature shape:', X_full.shape)  # 60 PCA + 50 handcrafted + 7 KNN features = 117
    return pca, scaler, knn_model, X_full

## Label Encoding

In [9]:
label_encoders = {}
targets = {}

for col in SCORE_COLS:
    unknown_scores = sorted(set(df[col].dropna().astype(float)) - set(VALID_SCORES))
    if unknown_scores:
        raise ValueError(f'{col} has invalid scores: {unknown_scores}')
    targets[col] = df[col].astype(float).map(SCORE_TO_CLASS).astype(int).values

for col in ['intent', 'task_type']:
    le = LabelEncoder()
    targets[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {list(le.classes_)}')

targets['reasoning_chain_detected'] = df['reasoning_chain_detected'].astype(bool).astype(int).values

tier_le = LabelEncoder()
tier_le.fit(['T1', 'T2', 'T3'])
label_encoders['tier'] = tier_le
tier_true_encoded = tier_le.transform(df['tier'].values)

# Output heads list
head_classes = {
    'intent': len(label_encoders['intent'].classes_),
    'task_type': len(label_encoders['task_type'].classes_),
    'reasoning_chain_detected': 2,
}
print('Standard heads:', list(head_classes.keys()))
print('Ordinal heads: d1, d2, d3, d4, d5')

intent: ['ANALYTICAL', 'FACTUAL', 'STRATEGIC', 'SYNTHETIC']
task_type: ['classification', 'coding', 'generation', 'reasoning', 'summarisation']
Standard heads: ['intent', 'task_type', 'reasoning_chain_detected']
Ordinal heads: d1, d2, d3, d4, d5


## Model Helpers — Ordinal & Balanced XGBoost

In [10]:
def make_xgb(num_classes, seed=RANDOM_STATE):
    is_binary = num_classes == 2
    params = dict(
        n_estimators=350,
        max_depth=3,
        learning_rate=0.045,
        subsample=0.88,
        colsample_bytree=0.88,
        reg_lambda=2.5,
        min_child_weight=5,
        random_state=seed,
        eval_metric='logloss' if is_binary else 'mlogloss',
        objective='binary:logistic' if is_binary else 'multi:softprob',
        tree_method='hist',
    )
    if not is_binary:
        params['num_class'] = num_classes
    return XGBClassifier(**params)


def capped_sample_weight(y, cap=3.0):
    weights = compute_sample_weight(class_weight='balanced', y=y)
    return np.clip(weights, 1.0 / cap, cap)


WEIGHT_CAPS = {
    'intent': 3.0,
    'task_type': 2.5,
    'reasoning_chain_detected': 2.0,
}


def fit_ordinal_head(name, X_train, y_train, seed=RANDOM_STATE):
    """Train 4 binary threshold models representing target progress: >=0.25, >=0.50, >=0.75, >=1.00"""
    models = []
    for threshold in range(1, 5):
        y_binary = (y_train >= threshold).astype(int)
        if len(np.unique(y_binary)) < 2:
            # Fallback for rare folds without class representation
            class_val = int(y_binary[0])
            models.append(class_val)
        else:
            model = make_xgb(num_classes=2, seed=seed + threshold)
            sw = capped_sample_weight(y_binary, cap=2.0)  # binary cap
            model.fit(X_train, y_binary, sample_weight=sw)
            models.append(model)
    return models


def predict_ordinal_proba(models, X):
    """Reconstruct 5 class probabilities from monotonic threshold model outputs"""
    N = len(X)
    p_gt = np.zeros((N, 4))
    for idx, model in enumerate(models):
        if isinstance(model, (int, float)):
            p_gt[:, idx] = model
        else:
            p_gt[:, idx] = model.predict_proba(X)[:, 1]

    probas = np.zeros((N, 5))
    probas[:, 0] = 1.0 - p_gt[:, 0]
    probas[:, 1] = p_gt[:, 0] - p_gt[:, 1]
    probas[:, 2] = p_gt[:, 1] - p_gt[:, 2]
    probas[:, 3] = p_gt[:, 2] - p_gt[:, 3]
    probas[:, 4] = p_gt[:, 3]

    probas = np.clip(probas, 0.0, 1.0)
    row_sums = probas.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return probas / row_sums


def train_all_heads(X_train, train_idx, seed=RANDOM_STATE):
    heads = {}
    # 1. Standard classification heads
    for name in head_classes.keys():
        y_train = targets[name][train_idx]
        heads[name] = fit_head(name, X_train, y_train, seed=seed)

    # 2. Ordinal dimension heads
    for col in SCORE_COLS:
        y_train = targets[col][train_idx]
        heads[col] = fit_ordinal_head(col, X_train, y_train, seed=seed)
    return heads


def fit_head(name, X_train, y_train, seed=RANDOM_STATE):
    model = make_xgb(head_classes[name], seed=seed)
    sw = capped_sample_weight(y_train, cap=WEIGHT_CAPS.get(name, 3.0))
    model.fit(X_train, y_train, sample_weight=sw)
    return model

## 5-Fold Stratified Cross-Validation

In [12]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Define the list of all prediction heads
ALL_HEADS = list(head_classes.keys()) + SCORE_COLS

fold_results = {name: {'accuracy': [], 'macro_f1': []} for name in ALL_HEADS + ['derived_tier']}

oof_true = {name: [] for name in ALL_HEADS + ['derived_tier']}
oof_pred = {name: [] for name in ALL_HEADS + ['derived_tier']}

slice_tier_results = {}
tier_confusions = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['tier']), start=1):
    print(f'\n=== Fold {fold}/{N_SPLITS} ===')

    # Fit PCA, scaler, and Cosine KNN model inside fold to prevent leakage
    fold_pca, fold_scaler, fold_knn, X_train = fit_shared_transformers(
        embeddings[train_idx], hand_df.iloc[train_idx].values, df.iloc[train_idx]
    )
    X_val = transform_shared_features(
        embeddings[val_idx], hand_df.iloc[val_idx].values, fold_pca, fold_scaler, fold_knn, df.iloc[train_idx]
    )

    # Train all heads
    heads = train_all_heads(X_train, train_idx, seed=RANDOM_STATE + fold)

    # Evaluate standard heads
    for name in head_classes.keys():
        y_true = targets[name][val_idx]
        y_pred = heads[name].predict(X_val)
        fold_results[name]['accuracy'].append(accuracy_score(y_true, y_pred))
        fold_results[name]['macro_f1'].append(f1_score(y_true, y_pred, average='macro', zero_division=0))
        oof_true[name].extend(y_true.tolist())
        oof_pred[name].extend(y_pred.tolist())

    # Evaluate ordinal dimension heads
    expected_dims = {}
    for col in SCORE_COLS:
        y_true = targets[col][val_idx]
        probas = predict_ordinal_proba(heads[col], X_val)
        y_pred = np.argmax(probas, axis=1)

        fold_results[col]['accuracy'].append(accuracy_score(y_true, y_pred))
        fold_results[col]['macro_f1'].append(f1_score(y_true, y_pred, average='macro', zero_division=0))
        oof_true[col].extend(y_true.tolist())
        oof_pred[col].extend(y_pred.tolist())

        # Expected score is probability weighted score value sum
        expected_dims[col] = probas @ np.array(VALID_SCORES)

    # Derive continuous complexity score and assign formula tier
    pred_scores = pd.DataFrame(expected_dims)
    complexity_scores = complexity_score_from_dims_frame(pred_scores)
    derived_tier_labels = complexity_scores.apply(tier_from_score).values
    derived_tier_encoded = tier_le.transform(derived_tier_labels)
    true_tier_encoded = tier_true_encoded[val_idx]

    tier_acc = accuracy_score(true_tier_encoded, derived_tier_encoded)
    tier_f1 = f1_score(true_tier_encoded, derived_tier_encoded, average='macro', zero_division=0)
    fold_results['derived_tier']['accuracy'].append(tier_acc)
    fold_results['derived_tier']['macro_f1'].append(tier_f1)
    oof_true['derived_tier'].extend(true_tier_encoded.tolist())
    oof_pred['derived_tier'].extend(derived_tier_encoded.tolist())

    tier_confusions.append(confusion_matrix(
        true_tier_encoded, derived_tier_encoded,
        labels=np.arange(len(tier_le.classes_))
    ))

    # Slice diagnostics - Using the correct column from the dataframe
    cs = df.iloc[val_idx]['computed_complexity_score'].values
    slices = {
        'all': np.ones(len(val_idx), dtype=bool),
        'boundary_t12': (cs >= 0.35) & (cs <= 0.44),
        'boundary_t23': (cs >= 0.65) & (cs <= 0.74),
        'safe': (cs < 0.30) | ((cs >= 0.45) & (cs <= 0.64)) | (cs >= 0.75),
    }
    for slice_name, mask in slices.items():
        if mask.sum() == 0:
            continue
        slice_tier_results.setdefault(slice_name, []).append(
            accuracy_score(true_tier_encoded[mask], derived_tier_encoded[mask])
        )

    print(f'Derived tier acc: {tier_acc:.4f}')


# === Summary ===
print('\n' + '=' * 72)
print('5-Fold CV Results (mean +/- std)')
print('=' * 72)
for name in ALL_HEADS + ['derived_tier']:
    acc = np.array(fold_results[name]['accuracy'])
    f1 = np.array(fold_results[name]['macro_f1'])
    print(f'{name:30s} Acc: {acc.mean():.4f} +/- {acc.std():.4f}   F1: {f1.mean():.4f} +/- {f1.std():.4f}')

print('\nDerived tier accuracy by slice:')
for slice_name, scores in slice_tier_results.items():
    scores = np.array(scores)
    print(f'{slice_name:14s} Acc: {scores.mean():.4f} +/- {scores.std():.4f}')

tier_labels = tier_le.classes_
print('\nAggregate derived tier confusion:')
print(pd.DataFrame(np.sum(tier_confusions, axis=0), index=tier_labels, columns=tier_labels))

print('\nDerived tier classification report:')
print(classification_report(oof_true['derived_tier'], oof_pred['derived_tier'], target_names=tier_labels, zero_division=0))
print('\nD1 classification report:')
print(classification_report(oof_true['d1'], oof_pred['d1'], target_names=[str(v) for v in VALID_SCORES], zero_division=0))
print('\nD2 classification report:')
print(classification_report(oof_true['d2'], oof_pred['d2'], target_names=[str(v) for v in VALID_SCORES], zero_division=0))
print('\nIntent classification report:')
print(classification_report(oof_true['intent'], oof_pred['intent'], target_names=label_encoders['intent'].classes_, zero_division=0))
print('\nTask type classification report:')
print(classification_report(oof_true['task_type'], oof_pred['task_type'], target_names=label_encoders['task_type'].classes_, zero_division=0))


=== Fold 1/5 ===
Derived tier acc: 0.7794

=== Fold 2/5 ===
Derived tier acc: 0.7831

=== Fold 3/5 ===
Derived tier acc: 0.7872

=== Fold 4/5 ===
Derived tier acc: 0.8120

=== Fold 5/5 ===
Derived tier acc: 0.8285

5-Fold CV Results (mean +/- std)
intent                         Acc: 0.7228 +/- 0.0183   F1: 0.7416 +/- 0.0158
task_type                      Acc: 0.7658 +/- 0.0116   F1: 0.6756 +/- 0.0164
reasoning_chain_detected       Acc: 0.9062 +/- 0.0106   F1: 0.8614 +/- 0.0186
d1                             Acc: 0.6807 +/- 0.0194   F1: 0.7043 +/- 0.0151
d2                             Acc: 0.7480 +/- 0.0196   F1: 0.6789 +/- 0.0249
d3                             Acc: 0.7910 +/- 0.0223   F1: 0.7307 +/- 0.0148
d4                             Acc: 0.8228 +/- 0.0129   F1: 0.7010 +/- 0.0214
d5                             Acc: 0.7551 +/- 0.0167   F1: 0.7328 +/- 0.0255
derived_tier                   Acc: 0.7980 +/- 0.0190   F1: 0.8061 +/- 0.0195

Derived tier accuracy by slice:
all            A

## Final Full-Dataset Training

In [13]:
final_pca, final_scaler, final_knn_model, X_full = fit_full_feature_matrix()
final_train_idx = np.arange(len(df))
heads = train_all_heads(X_full, final_train_idx, seed=RANDOM_STATE)

print('Final models trained:', list(heads.keys()))

PCA variance explained: 0.5982
Feature shape: (2421, 117)
Final models trained: ['intent', 'task_type', 'reasoning_chain_detected', 'd1', 'd2', 'd3', 'd4', 'd5']


## Rule-Based Research Signals

In [14]:
RESEARCH_SIGNAL_KEYWORDS = {
    'market_research': ['market', 'industry', 'trend', 'tam', 'sam', 'som'],
    'competitive_analysis': ['competitor', 'competitive', 'benchmark', 'rival'],
    'regulatory_compliance': ['regulation', 'regulatory', 'compliance', 'gdpr', 'hipaa', 'sox', 'eu ai act'],
    'security': ['security', 'vulnerability', 'threat', 'risk', 'iam', 'zero trust'],
    'cloud_infrastructure': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'spend', 'budget', 'showback', 'chargeback'],
    'devops': ['ci/cd', 'pipeline', 'deployment', 'sre', 'devops', 'observability'],
    'data_engineering': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai_governance': ['ai governance', 'llm', 'model risk', 'genai', 'guardrail'],
    'system_integration': ['integration', 'api', 'webhook', 'middleware'],
    'supply_chain': ['supply chain', 'inventory', 'procurement', 'logistics'],
    'hr_tech': ['hr', 'employee', 'workforce', 'talent', 'recruiting'],
    'vendor_analysis': ['vendor', 'rfi', 'rfp', 'procurement'],
}


def extract_research_signals(prompt, d4_score):
    if d4_score <= 0:
        return []
    text = str(prompt).lower()
    signals = []
    for signal, keywords in RESEARCH_SIGNAL_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            signals.append(signal)
    return signals if signals else ['external_research']

## Inference Function

In [15]:
# Subset dataset fields stored in bundle for KNN inference
train_df_subset = df[['tier', 'd1', 'd2']].copy()


def build_features_for_prompts(new_prompts):
    new_prompts_str = [str(p) for p in new_prompts]
    prefixed = ['Represent this sentence: ' + p for p in new_prompts_str]
    new_embeddings = embedding_model.encode(
        prefixed, batch_size=32, show_progress_bar=False, normalize_embeddings=True,
    )
    new_hand = handcrafted_features(new_prompts_str, phrasing_styles=None)
    # Apply full pipeline transforms using full dataset model configurations
    return transform_shared_features(new_embeddings, new_hand.values, final_pca, final_scaler, final_knn_model, train_df_subset)


def predict_prompt(prompt):
    X_one = build_features_for_prompts([prompt])

    # Predict D1-D5 ordinal expected values
    predicted_dims = {}
    dim_confidences = []
    for col in SCORE_COLS:
        probas = predict_ordinal_proba(heads[col], X_one)
        pred_class = int(np.argmax(probas, axis=1)[0])
        predicted_dims[col] = CLASS_TO_SCORE[pred_class]
        dim_confidences.append(float(np.max(probas)))

    # Compute expected continuous score
    expected_scores = {}
    for col in SCORE_COLS:
        probas = predict_ordinal_proba(heads[col], X_one)
        expected_scores[col] = float(probas @ np.array(VALID_SCORES))

    score = (
        expected_scores['d1'] * 0.35 +
        expected_scores['d2'] * 0.20 +
        expected_scores['d3'] * 0.20 +
        expected_scores['d4'] * 0.15 +
        expected_scores['d5'] * 0.10
    )
    tier = tier_from_score(score)

    # Predict standard classification heads
    intent = label_encoders['intent'].inverse_transform(heads['intent'].predict(X_one))[0]
    task_type = label_encoders['task_type'].inverse_transform(heads['task_type'].predict(X_one))[0]
    reasoning_chain = bool(int(heads['reasoning_chain_detected'].predict(X_one)[0]))

    # Confidence calculation: mean of key head max probabilities
    key_confidences = dim_confidences + [
        float(np.max(heads['intent'].predict_proba(X_one)[0])),
        float(np.max(heads['task_type'].predict_proba(X_one)[0])),
        float(np.max(heads['reasoning_chain_detected'].predict_proba(X_one)[0])),
    ]
    confidence = float(np.mean(key_confidences))

    result = {}
    for col in SCORE_COLS:
        result[col] = predicted_dims[col]
        result[f'{col}_label'] = DIMENSION_LABELS[col]

    result.update({
        'complexity_score': round(float(score), 4),
        'tier': tier,
        'intent': intent,
        'task_type': task_type,
        'reasoning_chain_detected': reasoning_chain,
        'research_signals': extract_research_signals(prompt, predicted_dims['d4']),
        'confidence': round(confidence, 4),
    })
    return result

In [16]:
sample_prompt = 'Design a multi-cloud GenAI governance architecture for a Fortune 500 company, including compliance risks and vendor evaluation criteria.'
print(json.dumps(predict_prompt(sample_prompt), indent=2))

{
  "d1": 1.0,
  "d1_label": "Semantic Complexity",
  "d2": 1.0,
  "d2_label": "Domain Specificity",
  "d3": 1.0,
  "d3_label": "Output Formality",
  "d4": 0.75,
  "d4_label": "Research Dependency",
  "d5": 0.75,
  "d5_label": "Context Requirement",
  "complexity_score": 0.9329,
  "tier": "T3",
  "intent": "STRATEGIC",
  "task_type": "reasoning",
  "reasoning_chain_detected": true,
  "research_signals": [
    "regulatory_compliance",
    "security",
    "cloud_infrastructure",
    "ai_governance",
    "vendor_analysis"
  ],
  "confidence": 0.899
}


## Export Bundle

In [17]:
import pickle
from google.colab import files

MODEL_BUNDLE_PATH = '/content/phase3_v3_model_bundle.pkl'

model_bundle = {
    'embedding_model_name': EMBEDDING_MODEL_NAME,
    'embedding_query_prefix': 'Represent this sentence: ',
    'pca': final_pca,
    'scaler': final_scaler,
    'knn_model': final_knn_model,
    'train_df_subset': train_df_subset,
    'heads': heads,
    'label_encoders': label_encoders,
    'class_to_score': CLASS_TO_SCORE,
    'dimension_labels': DIMENSION_LABELS,
    'valid_scores': VALID_SCORES,
    'score_cols': SCORE_COLS,
    'architecture': 'single_stage_ordinal_knn',
}

with open(MODEL_BUNDLE_PATH, 'wb') as f:
    pickle.dump(model_bundle, f)

print(f'Saved model bundle to {MODEL_BUNDLE_PATH}')
files.download(MODEL_BUNDLE_PATH)

Saved model bundle to /content/phase3_v3_model_bundle.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>